In [ ]:
# Cell 0: API keys / credentials
# National Gas Transmission's public MIPI web service does not require
# an API key. Placeholder kept for consistency with other notebooks in
# this toolkit.
NGT_API_KEY = None

# 02 — UK Gas Demand Ingestion (National Gas Transmission)

Since Brexit, UK gas transmission data is **not** published on the
ENTSOG Transparency Platform — it comes from National Gas
Transmission's (NGT, formerly National Grid Gas) own MIPI
("Market Information Provision Initiative") public web service.

Endpoint: `https://marketinformation.natgrid.co.uk/MIPIws-public/public/publicwebservice.asmx`

We use the `GetPublicationDataWM` operation to pull the
**"NTS Volume Offtaken, Actual, Total"** publication item, which is
NGT's published daily total system demand for the National
Transmission System (NTS), in mcm/day.

> **Note:** This client was built from documented MIPI conventions and
> a synthetic schema check (both direct requests to `natgrid.co.uk` and
> WebFetch against NGT/CRAN documentation pages returned HTTP 403 from
> this environment's network policy). The XML parser below is
> deliberately schema-tolerant (it detects the repeating record element
> generically rather than hard-coding field names), but the publication
> item name and exact field names should be verified against a live
> response before relying on this for production analysis.

In [ ]:
import time
import xml.etree.ElementTree as ET
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

In [ ]:
MIPI_BASE_URL = "https://marketinformation.natgrid.co.uk/MIPIws-public/public/publicwebservice.asmx"
DEMAND_PUBLICATION_ITEM = "NTS Volume Offtaken, Actual, Total"

DEFAULT_TIMEOUT = 30
MAX_RETRIES = 5
RETRY_BACKOFF_SECONDS = 2

In [ ]:
def _strip_ns(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def parse_mipi_xml_to_records(xml_text: str) -> list[dict]:
    """Flatten a MIPI XML response into a list of flat dicts.

    Rather than hard-coding an exact schema (unverified from this
    environment), this detects the most frequently repeated child tag
    anywhere in the document — that's the per-observation record — and
    flattens each into {leaf_tag: text}.
    """
    root = ET.fromstring(xml_text)
    tag_groups: dict[str, list[ET.Element]] = {}
    for parent in root.iter():
        children_by_tag: dict[str, list[ET.Element]] = {}
        for child in parent:
            children_by_tag.setdefault(_strip_ns(child.tag), []).append(child)
        for tag, els in children_by_tag.items():
            if len(els) > 1:
                tag_groups.setdefault(tag, []).extend(els)

    if not tag_groups:
        return []

    record_tag = max(tag_groups, key=lambda t: len(tag_groups[t]))
    records = []
    for el in tag_groups[record_tag]:
        record = {_strip_ns(leaf.tag): leaf.text for leaf in el.iter() if len(leaf) == 0}
        records.append(record)
    return records

In [ ]:
def fetch_mipi_publication(publication_item: str, start_date: date, end_date: date) -> pd.DataFrame:
    """Fetch a MIPI publication item over a date range, with retry/backoff."""
    params = {
        "PublicationObjectNameList": publication_item,
        "DateFrom": start_date.strftime("%d/%m/%Y"),
        "DateTo": end_date.strftime("%d/%m/%Y"),
        "LatestFlag": "N",
        "ApplicableForFlag": "Y",
    }
    url = f"{MIPI_BASE_URL}/GetPublicationDataWM"

    last_exc = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, params=params, timeout=DEFAULT_TIMEOUT)
            if resp.status_code == 429 or resp.status_code >= 500:
                raise requests.HTTPError(f"retryable status {resp.status_code}", response=resp)
            resp.raise_for_status()
            records = parse_mipi_xml_to_records(resp.text)
            return pd.DataFrame.from_records(records)
        except (requests.ConnectionError, requests.Timeout, requests.HTTPError, ET.ParseError) as exc:
            last_exc = exc
            if attempt == MAX_RETRIES - 1:
                break
            time.sleep(RETRY_BACKOFF_SECONDS * (2 ** attempt))
    raise RuntimeError(f"NGT MIPI request failed after {MAX_RETRIES} attempts: {url}") from last_exc

In [ ]:
def resolve_latest_ngt_date() -> date:
    """Probe MIPI for the most recent date with published demand data."""
    today = date.today()
    for lag in range(1, 8):
        candidate = today - timedelta(days=lag)
        df = fetch_mipi_publication(DEMAND_PUBLICATION_ITEM, candidate, candidate)
        if not df.empty:
            return candidate
    raise RuntimeError("No NGT MIPI demand data found in the last 7 days")


def resolve_analysis_date(analysis_date):
    if analysis_date is None:
        return resolve_latest_ngt_date()
    if isinstance(analysis_date, str):
        return datetime.strptime(analysis_date, "%Y-%m-%d").date()
    return analysis_date

In [ ]:
# ANALYSIS_DATE = None auto-resolves to the latest available NGT data point.
# Set an explicit date(YYYY, M, D) to pin the run to a specific end date instead.
ANALYSIS_DATE = None

LOOKBACK_DAYS = 365

resolved_date = resolve_analysis_date(ANALYSIS_DATE)
start_date = resolved_date - timedelta(days=LOOKBACK_DAYS)

print(f"ANALYSIS_DATE resolved to: {resolved_date}")
print(f"Fetching UK NTS demand {start_date} .. {resolved_date}")

In [ ]:
uk_demand_df = fetch_mipi_publication(DEMAND_PUBLICATION_ITEM, start_date, resolved_date)
uk_demand_df["country_code"] = "UK"
uk_demand_df.shape

In [ ]:
uk_demand_df.head()

In [ ]:
output_dir = Path.cwd().parent / "data"
output_dir.mkdir(exist_ok=True)
output_path = output_dir / f"uk_ngt_demand_{resolved_date.isoformat()}.parquet"

uk_demand_df.to_parquet(output_path, index=False)
print(f"Wrote {len(uk_demand_df)} rows to {output_path}")